# Neural Network Weight Initialisation — PDF, $E[X]$, and $\text{Var}(X)$

When training a neural network, the initial weight distribution determines whether the network can learn at all. Weights are random variables drawn from a chosen distribution.

The goal is to keep the **variance of activations stable** across all layers:

- If variance grows layer by layer → activations **explode** → gradients explode → training diverges.
- If variance shrinks layer by layer → activations **vanish** → gradients vanish → weights stop updating.

Two standard initialisations solve this:

**Xavier (Glorot) init** — designed for tanh / sigmoid activations:

$$
W \sim \text{Uniform}\!\left(-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}},\ \sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}\right)
$$

**He init** — designed for ReLU activations:

$$
W \sim N\!\left(0,\ \sqrt{\frac{2}{n_{\text{in}}}}\right)
$$

Both are derived by requiring $\text{Var}(\text{output}) = \text{Var}(\text{input})$ through one layer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Network: 6 layers, each with 512 neurons
layer_sizes = [512] * 7  # 6 weight matrices connecting 7 layers
n_samples = 1000

def relu(x):
    return np.maximum(0, x)

def forward_pass(x, layer_sizes, init_fn):
    """Returns mean and std of activations at each layer."""
    stats = [{"mean": x.mean(), "std": x.std()}]
    current = x
    for i in range(len(layer_sizes) - 1):
        n_in, n_out = layer_sizes[i], layer_sizes[i + 1]
        W = init_fn(n_in, n_out)
        current = relu(current @ W)
        stats.append({"mean": current.mean(), "std": current.std()})
    return stats

# Input: standard normal
x0 = rng.normal(0, 1, size=(n_samples, layer_sizes[0]))

## Step 1: Large random initialisation — activations explode

If weights are drawn from $N(0, 1)$ without scaling, the variance multiplies with each layer.

In [ ]:
def init_large(n_in, n_out):
    return rng.normal(0, 1, size=(n_in, n_out))

def init_small(n_in, n_out):
    return rng.normal(0, 0.01, size=(n_in, n_out))

def init_xavier(n_in, n_out):
    limit = np.sqrt(6 / (n_in + n_out))
    return rng.uniform(-limit, limit, size=(n_in, n_out))

def init_he(n_in, n_out):
    std = np.sqrt(2 / n_in)
    return rng.normal(0, std, size=(n_in, n_out))

stats_large = forward_pass(x0, layer_sizes, init_large)
stats_small = forward_pass(x0, layer_sizes, init_small)
stats_xavier = forward_pass(x0, layer_sizes, init_xavier)
stats_he = forward_pass(x0, layer_sizes, init_he)

## Step 2: Compare standard deviation of activations across layers

A well-initialised network keeps $\text{std}(\text{activation})$ roughly constant across all layers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = ["Large N(0,1)", "Small N(0,0.01)", "Xavier", "He"]
all_stats = [stats_large, stats_small, stats_xavier, stats_he]
colors = ["red", "orange", "steelblue", "green"]

layer_idx = range(len(layer_sizes))

for label, stats, color in zip(labels, all_stats, colors):
    stds = [s["std"] for s in stats]
    means = [s["mean"] for s in stats]
    axes[0].plot(layer_idx, stds, marker="o", label=label, color=color)
    axes[1].plot(layer_idx, means, marker="o", label=label, color=color)

axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Std dev of activations")
axes[0].set_title("Std dev across layers")
axes[0].legend()
axes[0].set_yscale("log")

axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Mean of activations")
axes[1].set_title("Mean across layers")
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 3: Plot activation distributions at layer 3

In [ ]:
# Re-run forward passes storing actual activations at layer 3
target_layer = 3

def activations_at_layer(x, layer_sizes, init_fn, target):
    current = x.copy()
    for i in range(len(layer_sizes) - 1):
        n_in, n_out = layer_sizes[i], layer_sizes[i + 1]
        W = init_fn(n_in, n_out)
        current = relu(current @ W)
        if i + 1 == target:
            return current.flatten()
    return current.flatten()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (label, init_fn, color) in zip(axes, [
    ("Large N(0,1)", init_large, "red"),
    ("Small N(0,0.01)", init_small, "orange"),
    ("Xavier", init_xavier, "steelblue"),
    ("He", init_he, "green"),
]):
    acts = activations_at_layer(x0, layer_sizes, init_fn, target_layer)
    # clip for display
    acts_clipped = np.clip(acts, -1e6, 1e6)
    ax.hist(acts_clipped, bins=60, density=True, color=color, alpha=0.7)
    ax.set_title(f"{label} — layer {target_layer}\nstd = {acts.std():.3g}")
    ax.set_xlabel("Activation value")
    ax.set_ylabel("Density")

plt.tight_layout()
plt.show()

## Step 4: Why does the He formula work?

For a ReLU layer with $n_{\text{in}}$ inputs:

$$
y = \text{ReLU}\!\left(\sum_{i=1}^{n_{\text{in}}} w_i x_i\right)
$$

If $w_i \sim N(0, \sigma_w^2)$ and $x_i$ are i.i.d. with $\text{Var}(x_i) = \sigma_x^2$, then before the ReLU:

$$
\text{Var}(z) = n_{\text{in}} \cdot \sigma_w^2 \cdot \sigma_x^2
$$

ReLU zeros out half the values, halving the variance:

$$
\text{Var}(y) \approx \frac{1}{2} n_{\text{in}} \cdot \sigma_w^2 \cdot \sigma_x^2
$$

Setting $\text{Var}(y) = \text{Var}(x) = \sigma_x^2$ gives:

$$
\sigma_w^2 = \frac{2}{n_{\text{in}}} \quad \Rightarrow \quad \sigma_w = \sqrt{\frac{2}{n_{\text{in}}}}
$$

In [ ]:
# Verify: with He init, Var(output) ≈ Var(input) at each layer
print("He init — std of activations at each layer:")
for i, s in enumerate(stats_he):
    print(f"  Layer {i}: std = {s['std']:.4f}")

## Observation

- Weight initialisation is a direct application of **variance control through a PDF**.
- The choice of distribution ($N$ vs $\text{Uniform}$) and its parameters determine $\text{Var}(W)$, which propagates through the network layer by layer.
- **Large init**: $\text{Var}(\text{activations})$ explodes exponentially — gradients become NaN.
- **Small init**: $\text{Var}(\text{activations})$ vanishes exponentially — all neurons output 0, gradients are zero, learning stops.
- **Xavier / He**: engineered so $\text{Var}(\text{output}) = \text{Var}(\text{input})$ at each layer — the variance neither explodes nor vanishes.
- The derivation of both formulas is a direct calculation of $E[X]$ and $\text{Var}(X)$ for products of random variables.